# Lesson 1.4: Your own client (retries, backoff, record/replay cache)
*Build an AI Agent from Scratch with LangGraph · Chapter 1*

**Goal:** when the network or the provider says "not now", send the same request again a little later, and never pay
twice for a call you already made.

**What you will be able to do**
- read the rate-limit headers and tell transient failures (dropped connection, 429, 5xx) from permanent ones (400, 401)
- compute exponential backoff with a cap and jitter by hand, and write `post_retry` yourself
- build a record/replay cache keyed by a hash of the request body, and know when it misses
- see why the course's Nightwatch client replays whole incidents for free

## How to use this notebook
- Run the cells **from top to bottom** (Shift + Enter).
- Cells that start with `# calls the API` send real requests to Groq: two small ones run by default, and one opt-in
  burst of 40 requests (`RUN_BURST = False`). Everything else uses a **fake server** in Python, so you can watch
  retries without waiting for a real outage. Without a key the API cells print `skipped`.
- Each exercise has a check cell that prints `✓ Exercise N passed`; no check cell calls the API. Solutions at the end.
- **Colab:** upload the `nightwatch/` folder too, and set the key with
  `from getpass import getpass; os.environ["GROQ_API_KEY"] = getpass("Groq key: ")`.

In [1]:
# Setup: run this cell first.
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))          # finds a .env in this folder or any folder above it
OFFLINE = os.environ.get("NOTEBOOK_OFFLINE") == "1"   # set NOTEBOOK_OFFLINE=1 to skip every API cell
HAVE_KEY = bool(os.environ.get("GROQ_API_KEY")) and not OFFLINE
MODEL = "openai/gpt-oss-20b"
if HAVE_KEY:
    print("GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.")
else:
    print("No GROQ_API_KEY: cells marked 'calls the API' print 'skipped' and the video's numbers are used.")
    print("Get the free key in lesson 0.2, put the line GROQ_API_KEY=... in a .env file, restart the kernel.")

GROQ_API_KEY found: cells marked 'calls the API' will really call Groq.


In [2]:
# Make the course code importable: find the repo's nightwatch/ folder above this notebook.
import sys
from pathlib import Path

here = Path.cwd()
NIGHTWATCH = next((p / "nightwatch" for p in [here, *here.parents]
                   if (p / "nightwatch" / "agent_v0.py").exists()), None)
if NIGHTWATCH:
    sys.path.insert(0, str(NIGHTWATCH))
    print("nightwatch/ found at", Path(os.path.relpath(NIGHTWATCH, here)).as_posix())
else:
    print("nightwatch/ not found: open this notebook from lessons/<lesson>/notebook/ inside the course repo.")

nightwatch/ found at ../../../nightwatch


In [3]:
import hashlib
import json
import random
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor

import httpx

URL = "https://api.groq.com/openai/v1/chat/completions"
HEADERS = {"Authorization": f"Bearer {os.environ.get('GROQ_API_KEY', '')}"}
FILLER = "Log line: checkout handled request in 120 ms. " * 60          # ~600 tokens, as in the video

def body(i):
    return {"model": MODEL, "temperature": 0, "max_completion_tokens": 64,
            "messages": [{"role": "user", "content": FILLER + f"\nRequest {i}: reply with the word OK."}]}

def http_send(b):
    """One real POST to Groq."""
    return httpx.post(URL, json=b, headers=HEADERS, timeout=60)

## 1. Recall (lesson 1.3)
When the model's JSON failed validation, we sent the error back and asked again: a **retry with feedback**.
Today the request was fine but **no answer came back at all**, so we send the very same request again, a little later.

In [4]:
print("lesson 1.3: bad answer   -> new request with the error attached")
print("lesson 1.4: no answer    -> the same request again, after a wait")

lesson 1.3: bad answer   -> new request with the error attached
lesson 1.4: no answer    -> the same request again, after a wait


## 2. Key words

| Word | Meaning |
|---|---|
| rate limit | a cap on requests and tokens you may use in a time window |
| 429 | the status code for hitting that cap: too many requests |
| retry-after | a header that may come with it: how many seconds to wait |
| exponential backoff | wait longer after each failure: 1 s, 2 s, 4 s, ... |
| jitter | a small random extra wait, so clients don't all retry at the same instant |
| idempotent | safe to send twice: the second copy changes nothing |
| record/replay cache | saves real responses and plays them back for the same request |

## 3. The idea and hand traces
**Transient or permanent?** Retry the first group, nothing else.

| Failure | Group | Retry? |
|---|---|---|
| dropped connection (`ConnectError`, no status at all) | transient | yes |
| 429 too many requests | transient | yes (use `retry-after` if sent) |
| 500 / 502 / 503 server error | transient | yes |
| 400 bad request | permanent | no: the same error eight times, only slower |
| 401 wrong key | permanent | no |

**Backoff by hand:** after failed attempt *a*, wait `min(30, 2**a)` seconds, plus jitter `uniform(0, 0.5)`.

| Attempt | 2**a | Capped wait | With jitter |
|---|---|---|---|
| 0 | 1 | 1 | 1.0 to 1.5 |
| 1 | 2 | 2 | 2.0 to 2.5 |
| 2 | 4 | 4 | 4.0 to 4.5 |
| 3 | 8 | 8 | 8.0 to 8.5 |
| 4 | 16 | 16 | 16.0 to 16.5 |
| 5 | 32 | **30** | 30.0 to 30.5 |

**Why jitter:** 20 clients fail together. Without jitter all 20 wake at exactly 1 s and hit the provider together
again; with jitter they spread over half a second: a trickle instead of a wave.

**Record/replay:** key = SHA-256 of the request body as JSON with sorted keys. File exists → replay. No file → one
real call, save it → record. Change one character → new key → a miss and a real, billed call.

In [5]:
for a in range(6):
    print(a, 2 ** a, min(30, 2 ** a))

0 1 1
1 2 2
2 4 4
3 8 8
4 16 16
5 32 30


## 4. Write it yourself
### 4.1 Read the wire
Every reply, even a normal 200, carries rate-limit headers.

In [6]:
# calls the API (1 request, ~800 tokens)
if HAVE_KEY:
    r = http_send(body(0))
    print("status:", r.status_code)
    for k in ("x-ratelimit-limit-requests", "x-ratelimit-remaining-requests", "x-ratelimit-limit-tokens",
              "x-ratelimit-remaining-tokens", "x-ratelimit-reset-tokens"):
        print(f"  {k}: {r.headers.get(k)}")
else:
    print("skipped (no key). Video: 200, requests 500000 / 499998 left, tokens 250000 / 248080 left, reset 460ms")

status: 200
  x-ratelimit-limit-requests: 500000
  x-ratelimit-remaining-requests: 499999
  x-ratelimit-limit-tokens: 250000
  x-ratelimit-remaining-tokens: 249194
  x-ratelimit-reset-tokens: 193ms


When *remaining* gets close to zero, slow down before the provider stops you. Going over gives a **429**, often
with `retry-after`. A dropped connection gives even less: no status, no headers, just an exception.

### 4.2 `post_retry` from scratch
Same loop as the video. Two extra parameters, `send` and `sleep`, let us plug in a fake server and skip real sleeping.

In [7]:
RETRY = (429, 500, 502, 503)

def post_retry(b, tries=8, send=http_send, sleep=time.sleep):
    for attempt in range(tries):
        try:
            r = send(b)
        except httpx.TransportError:          # dropped connection: no status code at all
            r = None
        if r is not None and r.status_code not in RETRY:
            return r, attempt                 # success, or a permanent error: hand it back
        ra = r.headers.get("retry-after") if r is not None else None
        wait = float(ra or 0) or min(30, 2 ** attempt)
        wait += random.uniform(0, 0.5)        # jitter: don't all wake up at once
        sleep(wait)
    return r, tries                           # gave up: the caller sees what failed

### 4.3 A fake server to test against
`ScriptedServer` plays a script: `"drop"` raises a `ConnectError`, a number returns that status, a `(429, "3")` tuple
adds `retry-after: 3`. After the script it answers 200 with a small chat completion. It counts every send.

In [8]:
FAKE_COMPLETION = {"id": "fake", "object": "chat.completion", "created": 0, "model": MODEL,
                   "choices": [{"index": 0, "finish_reason": "stop",
                                "message": {"role": "assistant", "content": "OK"}}],
                   "usage": {"prompt_tokens": 10, "completion_tokens": 1, "total_tokens": 11}}

class ScriptedServer:
    def __init__(self, script=()):
        self.script, self.sends = list(script), 0
    def __call__(self, b):
        self.sends += 1
        step = self.script.pop(0) if self.script else 200
        if step == "drop":
            raise httpx.ConnectError("connection dropped")
        code, ra = step if isinstance(step, tuple) else (step, None)
        return httpx.Response(code, headers={"retry-after": ra} if ra else {},
                              json=FAKE_COMPLETION if code == 200 else {"error": {"code": code}})

waits = []
server = ScriptedServer(["drop", 429, 503])
r, attempts = post_retry(body(1), send=server, sleep=waits.append)
print("status:", r.status_code, "| attempts before success:", attempts, "| sends:", server.sends)
print("waits:", [round(w, 2) for w in waits], "(1, 2, 4 seconds plus jitter)")

status: 200 | attempts before success: 3 | sends: 4
waits: [1.22, 2.08, 4.17] (1, 2, 4 seconds plus jitter)


### 4.4 Twenty calls, before and after
The video's burst, replayed on a fake server. Before: the same five requests drop their connection as in the video
(requests 5, 9, 10, 11, 12). After: a fresh burst where eight connections drop once each. No API calls, no real sleeping.

In [9]:
random.seed(7)
DROPS_BEFORE = {5: 1, 9: 1, 10: 1, 11: 1, 12: 1}    # request -> how many times its connection drops
DROPS_AFTER = {2: 1, 4: 1, 7: 1, 9: 1, 13: 1, 15: 1, 16: 1, 19: 1}

def burst_server(drops):
    left = dict(drops)
    def send(b):
        i = int(b["messages"][0]["content"].rsplit("Request ", 1)[1].split(":")[0])
        if left.get(i, 0) > 0:
            left[i] -= 1
            raise httpx.ConnectError("connection dropped")
        return httpx.Response(200, json=FAKE_COMPLETION)
    return send

def bare(i, send):
    try:
        return send(body(i)).status_code
    except httpx.TransportError as e:
        return type(e).__name__

send = burst_server(DROPS_BEFORE)
codes = [bare(i, send) for i in range(1, 21)]
print("before:", codes.count(200), "ok |", 20 - codes.count(200), "failed |", sorted({str(c) for c in codes if c != 200}))

send, waits = burst_server(DROPS_AFTER), []
res = [post_retry(body(i), send=send, sleep=waits.append) for i in range(1, 21)]
print("after: ", sum(r.status_code == 200 for r, _ in res), "of 20 ok | retries used:", sum(a for _, a in res),
      "| longest wait:", round(max(waits), 1), "s")

before: 15 ok | 5 failed | ['ConnectError']
after:  20 of 20 ok | retries used: 8 | longest wait: 1.3 s


Video, real Groq: before 15 OK and 5 `ConnectError`; after, 20 of 20 OK with 8 retries, 21.1 s in total,
longest wait 1.4 s. Five lost calls became zero for the price of eight short pauses. Bursts vary run to run (the live
coding run saw 12 OK and 8 failed).

In [10]:
# calls the API (opt-in: 40 requests, ~800 tokens each): set RUN_BURST = True to fire the real burst
RUN_BURST = False
if HAVE_KEY and RUN_BURST:
    with ThreadPoolExecutor(20) as ex:
        codes = list(ex.map(lambda i: bare(i, http_send), range(1, 21)))
    print("no retry:", codes.count(200), "ok of 20 |", sorted({str(c) for c in codes if c != 200}))
    t0 = time.time()
    with ThreadPoolExecutor(20) as ex:
        res = list(ex.map(lambda i: post_retry(body(100 + i)), range(20)))
    print("post_retry:", sum(r is not None and r.status_code == 200 for r, _ in res), "ok of 20 | retries:",
          sum(a for _, a in res), f"| {time.time() - t0:.1f} s")
else:
    print("skipped (RUN_BURST is False or no key)")

skipped (RUN_BURST is False or no key)


### 4.5 Record once, replay for free

In [11]:
def cache_key(b):
    return hashlib.sha256(json.dumps(b, sort_keys=True).encode()).hexdigest()[:16]

def cached_post(b, cache_dir, send=http_send):
    cache_dir.mkdir(exist_ok=True)
    f = cache_dir / f"{cache_key(b)}.json"
    if f.exists():
        return json.loads(f.read_text(encoding="utf-8")), "replay"
    r, _ = post_retry(b, send=send)
    f.write_text(r.text, encoding="utf-8")
    return r.json(), "record"

q = {"model": MODEL, "temperature": 0,
     "messages": [{"role": "user", "content": "Alert: checkout error rate is 23%. One sentence: what does it mean?"}]}
print("key of q:", cache_key(q))

key of q: 36da967736a14068


In [12]:
# calls the API (1 request: run 1 records, run 2 replays)
CACHE = Path("cache_demo")                   # next to this notebook
for f in CACHE.glob("*.json"):
    f.unlink()                               # start empty, like the video
send = http_send if HAVE_KEY else ScriptedServer()
if not HAVE_KEY:
    print("no key: a fake server stands in for Groq")
for run in (1, 2):
    t0 = time.time()
    data, how = cached_post(q, CACHE, send=send)
    ms = (time.time() - t0) * 1000
    tok = data["usage"]["total_tokens"] if how == "record" else 0
    print(f"run {run}: {how:6} {ms:8.1f} ms  tokens billed {tok:4}  -> {data['choices'][0]['message']['content'][:50]}...")
print("files in cache:", len(list(CACHE.glob("*.json"))))

run 1: record    436.3 ms  tokens billed  193  -> A 23 % checkout error rate means that roughly one ...
run 2: replay     11.8 ms  tokens billed    0  -> A 23 % checkout error rate means that roughly one ...
files in cache: 1


Video: record 460.1 ms and 193 tokens billed; replay 14.9 ms and 0 tokens, same text: about 30× faster, and free.
Your milliseconds will differ.

## 5. Library version + check: the SDK retries too
The `openai` SDK has its own retry loop (`max_retries`). We check it the same way, with a fake transport inside
`httpx`: no network, no key. It retries the 429s and does **not** retry a 400, just like `post_retry`.

In [13]:
import openai
from openai import OpenAI
print("openai", openai.__version__, "| httpx", httpx.__version__)

def sdk_with(script):
    server = ScriptedServer(script)
    def handler(request):
        try:
            resp = server(None)
        except httpx.ConnectError:
            raise httpx.ConnectError("connection dropped", request=request)
        resp.headers["retry-after-ms"] = "20"      # keep the demo fast
        return resp
    sdk = OpenAI(api_key="fake", base_url="https://api.groq.com/openai/v1", max_retries=6,
                 http_client=httpx.Client(transport=httpx.MockTransport(handler)))
    return sdk, server

msgs = [{"role": "user", "content": "hi"}]
sdk, server = sdk_with([429, 429])
resp = sdk.chat.completions.create(model=MODEL, messages=msgs)
print("two 429s then 200 ->", resp.choices[0].message.content, "| sends:", server.sends)

sdk, server = sdk_with([400])
try:
    sdk.chat.completions.create(model=MODEL, messages=msgs)
except openai.BadRequestError as e:
    print("400 ->", type(e).__name__, "| sends:", server.sends)
assert server.sends == 1
print("check: the SDK, like post_retry, sends a 400 only once")

openai 3.51.0 | httpx 0.28.1
two 429s then 200 -> OK | sends: 3
400 -> BadRequestError | sends: 1
check: the SDK, like post_retry, sends a 400 only once


## 6. Where it shows up in agents
Every step Nightwatch takes goes through one client, `agent_v0.chat`: retries for transient errors, and a
record/replay cache keyed by a SHA-256 of the body. In replay mode a whole incident runs from the cache, with no key
and no tokens. (**No API call.**)

In [14]:
if NIGHTWATCH:
    import inspect
    import agent_v0
    from brightcart.incidents import generate

    print(inspect.getdoc(agent_v0.chat))
    inc = next(x for x in generate() if x.id == "bad_deploy-2")
    t0 = time.time()
    try:
        run = agent_v0.run(inc, mode="replay")
        print(f"replayed {run['steps']} steps in {(time.time() - t0) * 1000:.0f} ms | usage recorded {run['usage']} | "
              f"tokens billed now: 0 | finish: {run['final']['cause']} on {run['final']['root_cause_service']}")
    except LookupError as e:
        print("not in this cache:", e)
else:
    print("nightwatch/ not found, skipped")

mode: record (cache hit or call+save), replay (cache only), live (always call, save).
replayed 9 steps in 5 ms | usage recorded {'prompt': 8198, 'completion': 941} | tokens billed now: 0 | finish: bad_deploy on web


## 7. Traps
**Trap 1: retrying a 400.** A bad request stays bad. A naive "retry everything" loop against a 400:

In [15]:
def retry_everything(b, send, tries=8):
    for attempt in range(tries):
        r = send(b)
        if r.status_code == 200:
            return r
    return r

naive = ScriptedServer([400] * 8)
retry_everything(body(1), naive)
ours = ScriptedServer([400] * 8)
post_retry(body(1), send=ours, sleep=lambda s: None)
try:
    assert naive.sends == 1, f"the naive loop sent the same bad request {naive.sends} times (post_retry: {ours.sends})"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: the naive loop sent the same bad request 8 times (post_retry: 1)


**Trap 2: no cap on retries.** A provider that is down keeps the agent waiting forever. Here a guard stops the
demo after 50 tries; the cap of 8 in `post_retry` returns the failure instead.

In [16]:
down = ScriptedServer([503] * 1000)
try:
    tries = 0
    while down(None).status_code != 200:     # no cap
        tries += 1
        if tries == 50:
            raise RuntimeError("still failing after 50 tries, and this loop would never stop on its own")
except RuntimeError as e:
    print("RuntimeError:", e)
r, n = post_retry(body(1), send=ScriptedServer([503] * 1000), sleep=lambda s: None)
print("post_retry gives up after", n, "tries and returns status", r.status_code)

RuntimeError: still failing after 50 tries, and this loop would never stop on its own
post_retry gives up after 8 tries and returns status 503


**Trap 3: no jitter.** Every client retries at the same instant and makes the next spike.

In [17]:
random.seed(1)
no_jitter = [min(30, 2 ** 0) for _ in range(20)]
jitter = [min(30, 2 ** 0) + random.uniform(0, 0.5) for _ in range(20)]
print("no jitter: 20 clients wake at", sorted(set(no_jitter)))
print(f"jitter:    spread from {min(jitter):.2f} s to {max(jitter):.2f} s")

no jitter: 20 clients wake at [1]
jitter:    spread from 1.00 s to 1.47 s


**Trap 4: retrying a write that is not idempotent.** The server restarted the service, then the connection
dropped before the reply arrived. The retry restarts it again (chapter 3 runs into exactly this bug).

In [18]:
restarts = []
def restart_service_endpoint(b):
    restarts.append(b["service"])            # the server applies the change ...
    if len(restarts) == 1:
        raise httpx.ConnectError("reply lost")   # ... but the reply never arrives
    return httpx.Response(200, json={"ok": True})

post_retry({"service": "checkout"}, send=restart_service_endpoint, sleep=lambda s: None)
try:
    assert len(restarts) == 1, f"checkout was restarted {len(restarts)} times"
except AssertionError as e:
    print("AssertionError:", e)

AssertionError: checkout was restarted 2 times


**Trap 5: caching a reply made with temperature above zero.** The cache freezes one sample and replays it as
"the" answer.

In [19]:
def sampler(b):
    word = random.choice(["outage", "timeout", "bad deploy", "DNS"])
    done = dict(FAKE_COMPLETION, choices=[{"index": 0, "finish_reason": "stop",
                                          "message": {"role": "assistant", "content": word}}])
    return httpx.Response(200, json=done)

hot = dict(q, temperature=1)
tmp = Path(tempfile.mkdtemp())
answers = [cached_post(hot, tmp, send=sampler) for _ in range(4)]
print([(how, d["choices"][0]["message"]["content"]) for d, how in answers])
print("fresh samples would differ:", [sampler(hot).json()["choices"][0]["message"]["content"] for _ in range(4)])

[('record', 'outage'), ('replay', 'outage'), ('replay', 'outage'), ('replay', 'outage')]
fresh samples would differ: ['DNS', 'timeout', 'DNS', 'outage']


## 8. Exercises
All checks use `ScriptedServer` or temp folders: no API.

In [20]:
def check(n, test):
    """Run the asserts in test(). Print one clear line instead of stopping the notebook."""
    try:
        test()
    except AssertionError as e:
        print(f"✗ Exercise {n} failed: {e}")
    except Exception as e:
        print(f"✗ Exercise {n} failed: {type(e).__name__}: {e}")
    else:
        print(f"✓ Exercise {n} passed")

### Exercise 1 (from the video): three steps
**(a)** Write `post_retry_logged`: `post_retry` plus a `log` list that gets every wait appended.
**(b)** A broken body: the server answers 400. Send it through `post_retry` and store how many sends it took in `broken_sends`.
**(c)** Change one word in the cached question `q` to make `q2`. Call `cached_post` for `q` and `q2` into `EX_CACHE`
with `send=ScriptedServer()`, and count the files in `n_files`.

In [21]:
def post_retry_logged(b, tries=8, send=http_send, sleep=time.sleep, log=None):
    # YOUR CODE HERE (a)
    return None, 0

bad_server = ScriptedServer([400])
# YOUR CODE HERE (b)
broken_sends = None

EX_CACHE = Path(tempfile.mkdtemp())
q2 = q
# YOUR CODE HERE (c)
n_files = None

In [22]:
def test_1():
    log = []
    r, a = post_retry_logged(body(1), send=ScriptedServer(["drop", 429, 200]), sleep=lambda s: None, log=log)
    assert r is not None and r.status_code == 200 and a == 2, f"(a) should succeed on attempt 2, got {a}"
    assert len(log) == 2 and 1 <= log[0] <= 1.5 and 2 <= log[1] <= 2.5, f"(a) waits should be ~1 and ~2 s: {log}"
    log = []
    post_retry_logged(body(1), send=ScriptedServer([(429, "3")]), sleep=lambda s: None, log=log)
    assert len(log) == 1 and 3 <= log[0] <= 3.5, f"(a) retry-after 3 should give a wait of 3 to 3.5 s: {log}"
    assert broken_sends == 1 and bad_server.sends == 1, f"(b) a 400 must be sent once, got {broken_sends}"
    assert q2 != q and q2["messages"][0]["content"] != q["messages"][0]["content"], "(c) change a word in q2's question"
    assert n_files == 2 == len(list(EX_CACHE.glob("*.json"))), f"(c) one changed word = a new key: expected 2 files, got {n_files}"

check(1, test_1)

✗ Exercise 1 failed: (a) should succeed on attempt 2, got 0


### Exercise 2: the wait, without jitter
Write `backoff_wait(attempt, retry_after=None)` that returns the wait in seconds **before jitter**: `retry_after`
(a string, as in the header) if given, else `2**attempt` capped at 30.

In [23]:
def backoff_wait(attempt, retry_after=None):
    # YOUR CODE HERE
    return 0

In [24]:
def test_2():
    got = [backoff_wait(a) for a in range(7)]
    assert got == [1, 2, 4, 8, 16, 30, 30], f"expected [1, 2, 4, 8, 16, 30, 30], got {got}"
    assert backoff_wait(0, "7") == 7.0, "use retry-after when the provider sends it"

check(2, test_2)

✗ Exercise 2 failed: expected [1, 2, 4, 8, 16, 30, 30], got [0, 0, 0, 0, 0, 0, 0]


### Exercise 3: transient or permanent?
Write `should_retry(result)`: `result` is a status code (int) or an exception. Return True for a transport error
(`httpx.TransportError`), 429, 500, 502, 503; False for everything else.

In [25]:
def should_retry(result):
    # YOUR CODE HERE
    return None

In [26]:
def test_3():
    yes = [httpx.ConnectError("x"), httpx.ReadTimeout("x"), 429, 500, 502, 503]
    no = [200, 400, 401, 404, ValueError("x")]
    assert all(should_retry(x) is True for x in yes), f"retry these: {[x for x in yes if should_retry(x) is not True]}"
    assert all(should_retry(x) is False for x in no), f"do not retry these: {[x for x in no if should_retry(x) is not False]}"

check(3, test_3)

✗ Exercise 3 failed: retry these: [ConnectError('x'), ReadTimeout('x'), 429, 500, 502, 503]


### Exercise 4: a key that ignores key order
Two bodies with the same content but keys written in a different order are the same request. Write `my_key(b)`
(SHA-256, first 16 hex characters) so that they get the same key, while one changed character gives a new key.

In [27]:
def my_key(b):
    # YOUR CODE HERE
    return ""

In [28]:
def test_4():
    a = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": "hi"}]}
    b = {"messages": [{"content": "hi", "role": "user"}], "temperature": 0, "model": MODEL}
    c = {"model": MODEL, "temperature": 0, "messages": [{"role": "user", "content": "hi!"}]}
    assert len(my_key(a)) == 16, "16 hex characters"
    assert my_key(a) == my_key(b), "same request, different key order: same key"
    assert my_key(a) != my_key(c), "one changed character: a new key"
    assert my_key(q) == cache_key(q), "should match cache_key from 4.5"

check(4, test_4)

✗ Exercise 4 failed: 16 hex characters


## Solutions (try first!)
### Solution 1

In [29]:
def post_retry_logged(b, tries=8, send=http_send, sleep=time.sleep, log=None):
    for attempt in range(tries):
        try:
            r = send(b)
        except httpx.TransportError:
            r = None
        if r is not None and r.status_code not in RETRY:
            return r, attempt
        ra = r.headers.get("retry-after") if r is not None else None
        wait = (float(ra or 0) or min(30, 2 ** attempt)) + random.uniform(0, 0.5)
        if log is not None:
            log.append(round(wait, 2))
        sleep(wait)
    return r, tries

bad_server = ScriptedServer([400])
r, _ = post_retry(body(1), send=bad_server, sleep=lambda s: None)
broken_sends = bad_server.sends

EX_CACHE = Path(tempfile.mkdtemp())
q2 = json.loads(json.dumps(q))
q2["messages"][0]["content"] = q2["messages"][0]["content"].replace("23%", "24%")
for b in (q, q2):
    cached_post(b, EX_CACHE, send=ScriptedServer())
n_files = len(list(EX_CACHE.glob("*.json")))

### Solution 2

In [30]:
def backoff_wait(attempt, retry_after=None):
    return float(retry_after) if retry_after else min(30, 2 ** attempt)

### Solution 3

In [31]:
def should_retry(result):
    if isinstance(result, BaseException):
        return isinstance(result, httpx.TransportError)
    return result in (429, 500, 502, 503)

### Solution 4

In [32]:
def my_key(b):
    return hashlib.sha256(json.dumps(b, sort_keys=True).encode()).hexdigest()[:16]